## Week 2 Day 3

ここからさらに詳しい内容に入っていきます。

1. 異なるモデル

2. Structured Outputs

3. Guardrails

In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

In [ ]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### OpenAI互換のエンドポイントを持つモデルなら、3ステップで簡単に使えます。

ステップ1: OpenAI互換のベースURLを見つける（guidesフォルダのGuide 9を参照）

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

ステップ2: Pythonクライアントライブラリのインスタンスを作成する（非同期版）

In [ ]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

ステップ3: モデルオブジェクトを作成する

In [ ]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [ ]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Kimi Sales Agent", instructions=instructions, model=kimi_model)
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [ ]:
from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [ ]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [ ]:
tools = [tool1, tool2, tool3, send_email_tool]

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=gemini_model)

In [ ]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

## Check out the trace

https://platform.openai.com/traces

## パート2: Structured Outputs

LLMは自然言語のテキストを生成します。しかし、その代わりに「pythonオブジェクト」を生成させることもできます。

これは、いつものやり方で実現されます。うまく作られたプロンプトとjsonです！

1. Pythonオブジェクトを指定します  
2. System promptの中で、LLMにJSONで応答するよう指示し、そのPythonオブジェクトを表すSchemaに従うよう指示します  
3. LLMはJSONを出力し、フレームワークがそれをもとにPythonオブジェクトを生成します

Pythonオブジェクトを指定する際には、Pydanticフレームワークの一部であるBaseModelのサブクラスを作成します。

Pydanticは、JSON schemaの定義とPython・json間のマッピングを簡単に行えるフレームワークです。

補足:

1. この仕組みには、実は本当に巧妙な部分があります。興味があれば「constrained decoding」を調べてみてください。
2. すべてのプロバイダーがStructured Outputsに対応しているわけではありません。


In [ ]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [ ]:
EmailReview.model_json_schema()

In [ ]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [ ]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model=gemini_model, output_type=EmailReview)
result = await Runner.run(checker, email)

In [ ]:
review = result.final_output
review

In [ ]:
review.is_professional

## パート3: Guardrails

Guardrailsは、Agentic AIにおいて非常に重要です。簡単に言えば、望ましくない挙動を防ぐために、ロジックとして、あるいは別のLLM呼び出しによってコーディングするコントロールのことです。

私にとって、OpenAI Agents SDKにおけるGuardrailsの実装は、少し「フレームワーク的な魔法」のように感じられます。おそらく、この重要なトピックに対処するためのフレームワークレベルのコントロールを示すことが、彼らの意図だったのだと思います。

しかし、Guardrailsを明示的に実装するのは、別個のRunner.run()呼び出しとして、あるいはツールの実装内でのチェックとして行う方がシンプルで綺麗です。

とはいえ、まずはフレームワークが提供するツーリングを見てみましょう。

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">要注意ポイント</h2>
            <span style="color:#ff7800;">OpenAI Agents SDKには3種類のGuardrailsがあります。input、output、toolです。Input guardrailsは、Runner.run()内で最初のAgentへの最初の入力に対してのみ実行されます。Output guardrailsは、最後のAgentの最終出力に対してのみ実行されます。他のAgentにguardrailsを設定しても、それらは決して呼び出されません。
            </span>
        </td>
    </tr>
</table>

In [ ]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [ ]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[email_guardrail])

In [ ]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

Check out the trace:

https://platform.openai.com/traces

## 一方で……

言うまでもないことですが、こちらの方がシンプルで、どのフレームワークでも動作します

In [ ]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)


In [ ]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

## traceを確認してみましょう。

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">・異なるモデルを試してみましょう<br/>・input guardrailsとoutput guardrailsをさらに追加してみましょう<br/>・メール生成にstructured outputsを使ってみましょう
            </span>
        </td>
    </tr>
</table>

## 補足オプション: Sandbox Agents

この例は、Windows + WSL2、Mac、Linuxでのみ動作します

https://openai.github.io/openai-agents-python/sandbox_agents/

これは実行用のハーネス、つまりランタイムです。「大規模なドキュメント群を検索したり、ファイルを編集したり、コマンドを実行したり、アーティファクトを生成したり、保存されたサンドボックスの状態から作業を再開したりできる、永続的なワークスペース」です。

以下を設定する必要があります。
1. Manifest: ワークスペース
2. Capabilities: 何ができるか
3. SandboxRunConfig: どこで実行するか

In [ ]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [ ]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [ ]:
CODE_DIR

In [ ]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [ ]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

In [ ]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [ ]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model=gemini_model, default_manifest=manifest, capabilities=capabilities)

In [ ]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

## 補足オプション: MCPの予告編！

In [ ]:
from agents.mcp import MCPServerStreamableHttp


In [ ]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
agent = Agent(name="Expert", instructions="Answer the question", model=gemini_model)
result = await Runner.run(agent, task)
print(result.final_output)

In [ ]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model=gemini_model)
    result = await Runner.run(agent, task)

print(result.final_output)

そしてtraceを見てみましょう。

https://platform.openai.com/traces
